In [1]:
import csv
import os
from io import StringIO
from typing import Any, cast

import pandas as pd
from azure.ai.formrecognizer import FormRecognizerClient
from azure.cognitiveservices.vision.customvision.prediction import CustomVisionPredictionClient
from azure.core.credentials import AzureKeyCredential
from azure.storage.blob import BlobClient, ContentSettings
from dotenv import load_dotenv
from msrest.authentication import ApiKeyCredentials

In [2]:
load_dotenv("../.env")

FLIGHT_MANIFEST_URL = os.environ["FLIGHT_MANIFEST_URL"]
ID_FORM_RECOGNIZER_ENDPOINT = os.environ["ID_RECOGNIZER_ENDPOINT"]
ID_FORM_RECOGNIZER_KEY = os.environ["ID_RECOGNIZER_KEY"]

BOARDING_PASS_RECOGNIZER_ENDPOINT = os.environ["BOARDING_PASS_RECOGNIZER_ENDPOINT"]
BOARDING_PASS_RECOGNIZER_KEY = os.environ["BOARDING_PASS_RECOGNIZER_KEY"]
BOARDING_PASS_TRAINING_URL = os.environ["BOARDING_PASS_TRAINING_URL"]
BOARDING_MODEL_ID = os.environ["BOARDING_MODEL_ID"]

publish_iteration_name = "boarding-kiosk-lighter-detector"
LIGHTER_DETECTOR_PROJECT_ID = os.environ["LIGHTER_DETECTOR_PROJECT_ID"]
LIGHTER_DETECTOR_PREDICTION_ENDPOINT = os.environ["LIGHTER_DETECTOR_PREDICTION_ENDPOINT"]
LIGHTER_DETECTOR_PREDICTION_KEY = os.environ["LIGHTER_DETECTOR_PREDICTION_KEY"]
LIGHTER_DETECTOR_PREDICTION_RESOURCE_ID = os.environ["LIGHTER_DETECTOR_PREDICTION_RESOURCE_ID"]

In [3]:
BOARDING_FIELDS = ["Flight No", "Passenger Name", "From", "Carrier", "Date", "To", "Seat", "Class", "Boarding Time"]

ID_ROOT_PATH = '../material_preparation_step/digital-ids'
BOARDING_ROOT_PATH = '../material_preparation_step/boarding_passes'
BAGGAGE_ROOT_PATH = '../material_preparation_step/baggage'

TRAVELLER_FILES = [('ca-dl-avkash-chauhan.png', 'boarding-avkash.pdf', None),
                   ('ca-dl-james-jackson.png', 'boarding-james.pdf', 'baggage-james.jpg'),
                   ('ca-dl-james-webb.png', 'boarding-james-webb.pdf', None),
                   ('ca-dl-libby-herold.png', 'boarding-libby.pdf', None),
                   ('ca-dl-radha-s-kumar.png', 'boarding-radha-s-kumar.pdf', 'baggage-radha.jpg')]

BOARDING_MISMATCH_MESSAGE = (
    "Dear {passenger_name},\n"
    "Some of the information in your boarding pass does not match the flight manifest data, so you cannot board the plane.\n"
    "Please see a customer service representative."
)

ID_MISMATCH_MESSAGE = (
    "Dear {passenger_name},\n"
    "Some of the information on your ID card does not match the flight manifest data, so you cannot board the plane.\n"
    "Please see a customer service representative."
)

In [4]:
def get_flight_manifest():
    blob_client = BlobClient.from_blob_url(FLIGHT_MANIFEST_URL)
    blob_data = blob_client.download_blob()
    csv_content = blob_data.readall().decode("utf-8")

    return pd.DataFrame(list(csv.DictReader(StringIO(csv_content))))

In [5]:
def extract_id_card_fields(identity_card):
    extracted_fields = {}
    for field_key in identity_card.fields.keys():
        extracted_fields[field_key] = identity_card.fields[field_key].value

    return extracted_fields


def extract_id_info(image_name: str):
    form_recognizer_client = FormRecognizerClient(endpoint=ID_FORM_RECOGNIZER_ENDPOINT,
                                                  credential=AzureKeyCredential(ID_FORM_RECOGNIZER_KEY))

    local_image_path = ID_ROOT_PATH + "/" + image_name
    with open(local_image_path, "rb") as f:
        poller = form_recognizer_client.begin_recognize_identity_documents(identity_document=f)
        return extract_id_card_fields(poller.result()[0])


In [6]:
def extract_board_passing(file_name):
    form_recognizer_client = FormRecognizerClient(endpoint=BOARDING_PASS_RECOGNIZER_ENDPOINT,
                                                  credential=AzureKeyCredential(BOARDING_PASS_RECOGNIZER_KEY))

    local_image_path = BOARDING_ROOT_PATH + "/" + file_name
    with open(local_image_path, "rb") as f:
        custom_test_action = form_recognizer_client.begin_recognize_custom_forms(model_id=BOARDING_MODEL_ID, form=f)
        custom_test_action_result = custom_test_action.result()

        result = extract_id_card_fields(custom_test_action_result[0])
        for field in BOARDING_FIELDS:
            result[field] = custom_test_action_result[0].fields.get(field).value if custom_test_action_result[
                0].fields.get(field) else None

        # Expand short class codes to match manifest values.
        class_value = result.get("Class")
        if isinstance(class_value, str):
            class_code = class_value.strip().upper()
            if class_code == "E":
                result["Class"] = "Economy"
            elif class_code == "B":
                result["Class"] = "Business"

        return result


In [7]:
prediction_credentials = cast(Any, ApiKeyCredentials(in_headers={"Prediction-key": LIGHTER_DETECTOR_PREDICTION_KEY}))
predictor = CustomVisionPredictionClient(endpoint=LIGHTER_DETECTOR_PREDICTION_ENDPOINT,
                                         credentials=prediction_credentials)


def detect_lighter(image_name):
    if image_name is None:
        return False

    local_image_path = os.path.join(BAGGAGE_ROOT_PATH, image_name)
    if not os.path.exists(local_image_path):
        return False

    with open(local_image_path, "rb") as image_contents:
        results = predictor.detect_image(
            LIGHTER_DETECTOR_PROJECT_ID,
            publish_iteration_name,
            image_contents.read()
        )

    for prediction in results.predictions:
        if prediction.probability > 0.75:
            return True

    return False

In [8]:
df_manifest: pd.DataFrame = get_flight_manifest()

In [9]:
def normalize_text(value):
    if value is None:
        return ""
    return " ".join(str(value).strip().lower().split())


def normalize_date(value):
    if value is None:
        return None
    parsed = pd.to_datetime(str(value).strip(), errors="coerce")
    if pd.isna(parsed):
        return None
    return parsed.date()


def get_boarding_dob(boarding_fields):
    # Support common DOB labels used by custom form models.
    for key in ["DateOfBirth", "Date of Birth", "DOB", "BirthDate"]:
        if key in boarding_fields and boarding_fields[key] is not None:
            return boarding_fields[key]
    return None


def format_passenger_name(first_name, last_name):
    return f"{str(first_name).strip().title()} {str(last_name).strip().title()}".strip()


def print_validation_message(validation_result, manifest_row) -> None:
    identity_validated = validation_result["NameValidation"] and validation_result["DoBValidation"]
    boarding_pass_validated = validation_result["BoardingPassValidation"]
    lighter_found = validation_result["LuggageValidation"]
    passenger_name = "Sir/Madam"

    if manifest_row is not None:
        passenger_name = format_passenger_name(manifest_row.get("First Name"), manifest_row.get("Last Name"))

    if lighter_found and identity_validated and boarding_pass_validated and manifest_row is not None:
        print(
            f"Dear Mr. {passenger_name},\n"
            f"You are welcome to flight # {manifest_row.get('Flight No.', '')} leaving at {manifest_row.get('Boarding Time', '')} "
            f"from {manifest_row.get('From (Origin)', '')} to {manifest_row.get('To (Destination)', '')}.\n"
            f"Your seat number is {manifest_row.get('Seat', '')}, and it is confirmed.\n"
            "We have found a prohibited item in your carry-on baggage, and it is flagged for removal.\n\n"
            "Your identity is verified. However, your baggage verification failed, so please see a customer service representative."
        )
    elif identity_validated and not boarding_pass_validated:
        print(BOARDING_MISMATCH_MESSAGE.format(passenger_name=passenger_name))
    elif manifest_row is None or (boarding_pass_validated and not identity_validated):
        print(ID_MISMATCH_MESSAGE.format(passenger_name=passenger_name))
    else:
        print(
            f"Dear Mr. {passenger_name},\n"
            f"You are welcome to flight # {manifest_row.get('Flight No.', '')} leaving at {manifest_row.get('Boarding Time', '')} "
            f"from {manifest_row.get('From (Origin)', '')} to {manifest_row.get('To (Destination)', '')}.\n"
            f"Your seat number is {manifest_row.get('Seat', '')}, and it is confirmed.\n"
            "Your identity and boarding pass are verified. We wish you a pleasant flight!"
        )

    print("==============================================================================================")


def validate_info(traveller):
    id_info = extract_id_info(traveller[0])
    boarding_info = extract_board_passing(traveller[1])

    id_first = normalize_text(id_info.get("FirstName"))
    id_last = normalize_text(id_info.get("LastName"))
    passenger_name = normalize_text(boarding_info.get("Passenger Name"))

    id_dob = normalize_date(id_info.get("DateOfBirth"))

    manifest_match = df_manifest[
        (df_manifest["First Name"].astype(str).str.strip().str.lower() == id_first)
        & (df_manifest["Last Name"].astype(str).str.strip().str.lower() == id_last)
        ]
    manifest_dob = None
    if not manifest_match.empty:
        manifest_dob = normalize_date(manifest_match.iloc[0]["Date of Birth"])

    # Name is valid only when boarding and ID names match and the person exists in manifest.
    name_validation = passenger_name == normalize_text(f"{id_first} {id_last}") and not manifest_match.empty

    # Date of birth is valid only when ID DoB matches the manifest record.
    dob_validation = id_dob is not None and manifest_dob is not None and id_dob == manifest_dob

    boarding_pass_validation = False
    if not manifest_match.empty:
        manifest_row = manifest_match.iloc[0]

        flight_no_match = normalize_text(
            f'{boarding_info.get("Carrier")}-{boarding_info.get("Flight No")}') == normalize_text(
            manifest_row.get("Flight No."))
        seat_match = normalize_text(boarding_info.get("Seat")) == normalize_text(manifest_row.get("Seat"))
        class_match = normalize_text(boarding_info.get("Class")) == normalize_text(manifest_row.get("Class"))
        origin_match = normalize_text(boarding_info.get("From")) == normalize_text(manifest_row.get("From (Origin)"))
        destination_match = normalize_text(boarding_info.get("To")) == normalize_text(
            manifest_row.get("To (Destination)"))
        date_match = normalize_date(boarding_info.get("Date")) == normalize_date(manifest_row.get("Flight Date"))

        boarding_time = normalize_text(boarding_info.get("Boarding Time"))
        manifest_boarding_time = normalize_text(manifest_row.get("Boarding Time"))
        time_match = boarding_time != "" and manifest_boarding_time.startswith(boarding_time)

        boarding_pass_validation = all([
            flight_no_match,
            seat_match,
            class_match,
            origin_match,
            destination_match,
            date_match,
            time_match,
        ])

    return {
        "NameValidation": bool(name_validation),
        "DoBValidation": bool(dob_validation),
        "BoardingPassValidation": bool(boarding_pass_validation),
        "LuggageValidation": not bool(detect_lighter(traveller[2]))
    }


In [10]:
def update_manifest_with_validations(travellers) -> pd.DataFrame:
    for traveller in travellers:
        validation_result = validate_info(traveller)

        id_info = extract_id_info(traveller[0])
        id_first = normalize_text(id_info.get("FirstName"))
        id_last = normalize_text(id_info.get("LastName"))

        manifest_mask = (
                (df_manifest["First Name"].astype(str).str.strip().str.lower() == id_first)
                & (df_manifest["Last Name"].astype(str).str.strip().str.lower() == id_last)
        )

        manifest_row = df_manifest.loc[manifest_mask].iloc[0] if manifest_mask.any() else None
        print_validation_message(validation_result, manifest_row)

        if manifest_mask.any():
            df_manifest.loc[manifest_mask, "NameValidation"] = (str(validation_result["NameValidation"])).upper()
            df_manifest.loc[manifest_mask, "DoBValidation"] = (str(validation_result["DoBValidation"])).upper()
            df_manifest.loc[manifest_mask, "BoardingPassValidation"] = (
                str(validation_result["BoardingPassValidation"])).upper()
            df_manifest.loc[manifest_mask, "LuggageValidation"] = (
                str(validation_result["LuggageValidation"])).upper()

    return df_manifest


updated_manifest: pd.DataFrame = update_manifest_with_validations(TRAVELLER_FILES)

Dear Sir/Madam,
Some of the information on your ID card does not match the flight manifest data, so you cannot board the plane.
Please see a customer service representative.
Dear Mr. James Jackson,
You are welcome to flight # UA-234 leaving at 10:00 AM PST from San Francisco to Chicago.
Your seat number is 25B, and it is confirmed.
Your identity and boarding pass are verified. We wish you a pleasant flight!
Dear Mr. James Webb,
You are welcome to flight # UA-234 leaving at 10:00 AM PST from San Francisco to Chicago.
Your seat number is 1A, and it is confirmed.
We have found a prohibited item in your carry-on baggage, and it is flagged for removal.

Your identity is verified. However, your baggage verification failed, so please see a customer service representative.
Dear Mr. Libby Herold,
You are welcome to flight # UA-234 leaving at 10:00 AM PST from San Francisco to Chicago.
Your seat number is 3D, and it is confirmed.
We have found a prohibited item in your carry-on baggage, and it i

In [11]:
def upload_updated_manifest(manifest_df: pd.DataFrame) -> None:
    blob_client = BlobClient.from_blob_url(FLIGHT_MANIFEST_URL)
    manifest_csv = manifest_df.to_csv(index=False).encode("utf-8")
    blob_client.upload_blob(
        manifest_csv,
        overwrite=True,
        content_settings=ContentSettings(content_type="text/csv"),
    )


upload_updated_manifest(updated_manifest)

In [12]:
updated_manifest

,Carrier,Flight No.,Class,From (Origin),To (Destination),Flight Date,Baggage,Seat,Gate,Boarding Time,Ticket No.,First Name,Last Name,Date of Birth,Sex,DoBValidation,PersonValidation,LuggageValidation,NameValidation,BoardingPassValidation
0,UA,UA-234,Economy,San Francisco,Chicago,April 20 2022,YES,34A,G1,10:00 AM PST,34236746,Sameer,Kumar,25-Jan-1990,M,FALSE,FALSE,FALSE,FALSE,FALSE
1,UA,UA-234,Economy,San Francisco,Chicago,April 20 2022,YES,34B,G1,10:00 AM PST,34236747,Radha,Kumar,5-Mar-1994,F,FALSE,FALSE,FALSE,FALSE,FALSE
2,UA,UA-234,Business,San Francisco,Chicago,April 20 2022,NO,1A,G1,10:00 AM PST,34236748,James,Webb,15-Dec-1970,M,TRUE,TRUE,TRUE,TRUE,TRUE
3,UA,UA-234,Business,San Francisco,Chicago,April 20 2022,NO,3D,G1,10:00 AM PST,34236749,Libby,Herold,10-Feb-1996,F,TRUE,TRUE,TRUE,TRUE,TRUE
4,UA,UA-234,Economy,San Francisco,Chicago,April 20 2022,YES,25B,G1,10:00 AM PST,34236750,James,Jackson,12-Oct-1956,M,TRUE,TRUE,FALSE,TRUE,TRUE
5,UA,UA-234,Economy,San Francisco,Chicago,April 20 2022,NO,20A,G1,10:00 AM PST,34236751,Avkash,Chauhan,1-Jan-1990,M,FALSE,FALSE,FALSE,FALSE,FALSE


In [13]:
updated_manifest.to_csv("../step_5/updated_manifest.csv", index=False)